**[🏠 Course Home](../README.md)** | **📖 [Conceptual Track](../conceptual/14_marginal_likelihood_and_model_comparison.ipynb)** | **🤖 [AI Track](14_marginal_likelihood_and_model_comparison.ipynb)** | **🐍 [Python Track](../python/14_marginal_likelihood_and_model_comparison.ipynb)** | **🔵 [R Track](../r/14_marginal_likelihood_and_model_comparison.ipynb)** | **⬅️ [Prev: Ch 13: Hierarchical Models](13_hierarchical_models_and_partial_pooling.ipynb)** | **➡️ [Next: Ch 15: Bayesian Decay & Memory](15_calibrating_bayesian_decay_and_memory.ipynb)**

*🧭 Part III: Applied Modeling & Hierarchy — Chapter 14 of 22*

---

# Chapter 14: Marginal Likelihood & Model Comparison
### *Prompting for Occam's Razor, Lindley's Paradox, and Robust PSIS-LOO*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
>
> When data science and engineering teams compare competing statistical
> models—such as choosing between a simple quadratic latency model and a
> flexible deep network or high-degree polynomial—they require objective
> arbitration.
>
> Large Language Models (LLMs) suffer from two opposite, equally lethal
> cognitive biases when tasked with model selection:
> 1. **Training Fit Greed**: Unconstrained AI coding assistants prioritize
>    in-sample metrics ($R^2$, training loss, training log-likelihood). They
>    routinely recommend overparameterized models (e.g., degree-7 polynomials)
>    that memorize training noise and explode in production.
> 2. **Lindley's Paradox Hallucination**: When instructed to perform Bayesian
>    hypothesis testing or compute Bayes Factors ($BF_{12}$), LLMs blindly
>    insert "uninformative" diffuse priors (e.g., $\beta \sim \mathcal{N}(0,
>    1000^2)$). This dilutes the prior probability budget across infinite
>    space, causing the Bayes Factor to spuriously reject true physical
>    mechanisms in favor of the null model.
>
> This module teaches you how to construct declarative contracts that balance
> Bayesian Occam's Razor with out-of-sample Pareto-smoothed cross-validation
> (PSIS-LOO).


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt That Fools the AI

A performance engineer wants to identify the best mathematical model for
database query latency as a function of system memory utilization. They want to
know whether a simple linear model, a quadratic model, or a high-degree
polynomial provides the best fit, and they ask for a formal Bayes Factor test:

```text
Compare whether a linear, quadratic, or degree-7 polynomial regression best
fits our database throughput telemetry. Compute R-squared and training
log-likelihood. Then compute Bayes Factors using standard non-informative
priors Normal(0, 1000^2) on the polynomial weights to test if higher degrees
are necessary.
```

---

### The Plausible AI Hallucination

The AI provides code that fits all three models and delivers this analysis:

1. **The Overfitting Recommendation**:
   > *"Model Selection Report: The Degree-7 Polynomial model achieves a
   > stellar training $R^2 = 0.994$ and training log-likelihood of **-112.4**,
   > vastly outperforming the Quadratic model ($R^2 = 0.941$, LL = -136.2).
   > The Degree-7 model captures fine-grained fluctuations that simpler models
   > miss!"*

2. **The Lindley Paradox Contradiction**:
   > *"However, when computing Bayes Factors with diffuse priors $\mathcal{N}(0,
   > 1000^2)$, we find $BF_{01} = 2.4 \times 10^4$ in favor of the Linear model
   > over the Quadratic model. Therefore, the Bayesian evidence decisively
   > proves that non-linear effects do not exist!"*

---

### Forensic Post-Mortem: Why the System Crashes

```
           THE PROBABILITY BUDGET & LINDLEY'S PARADOX
 Probability
   Density
    1.0 |      * [Calibrated Prior: N(0, 2^2)]
        |     / \   Tight probability budget concentrated
        |    /   \  on plausible physical effects!
    0.5 |   /     \
        |  /       \
    0.0 |-+---------+----------------------------------------------+---
         -5         0                                            +1000
        |----------------------------------------------------------|
                 Diffuse Prior: N(0, 1000^2) (Lindley Trap!)
           Probability density is diluted across empty space!
           Occam Penalty = (Posterior Volume / Prior Volume) -> 0
```

#### 1. The Flaw of Maximum Likelihood (In-Sample Greed)
Maximum likelihood estimation (and training $R^2$) rewards complexity
monotonically. A degree-7 polynomial has 8 free parameters. It bends to
accommodate every random spike in training data. When deployed on unseen
production queries, its out-of-sample error explodes by $400\%$.

#### 2. The Analytical Anatomy of the Occam Factor
Bayesian model comparison computes the **Marginal Likelihood** (or Evidence) by
integrating over the entire prior parameter space:
$$P(\mathcal{D} \mid \mathcal{M}) = \int P(\mathcal{D} \mid \boldsymbol{\theta}, \mathcal{M}) P(\boldsymbol{\theta} \mid \mathcal{M}) \, d\boldsymbol{\theta}$$
Under the Laplace approximation, this decomposes into:
$$P(\mathcal{D} \mid \mathcal{M}) \approx \underbrace{P(\mathcal{D} \mid \hat{\boldsymbol{\theta}}, \mathcal{M})}_{\text{Peak Likelihood}} \times \underbrace{\frac{\Delta \boldsymbol{\theta}_{\text{post}}}{\Delta \boldsymbol{\theta}_{\text{prior}}}}_{\text{Occam Factor}}$$
The **Occam Factor** represents the fraction of prior parameter volume that
survives into the posterior. If a model adds parameters that expand prior volume
without increasing likelihood, it suffers an exponential penalty.

#### 3. Lindley's Paradox: The Danger of "Vague" Priors
When the AI sets $\beta \sim \mathcal{N}(0, 1000^2)$, the prior width $\Delta
\beta_{\text{prior}}$ is $2,000$. The posterior width $\Delta
\beta_{\text{post}}$ is roughly $0.2$.
The Occam Factor is:
$$\frac{\Delta \beta_{\text{post}}}{\Delta \beta_{\text{prior}}} \approx \frac{0.2}{2000} = 10^{-4}$$
By making the prior arbitrarily wide, $P(\mathcal{D} \mid \mathcal{M}_2)$ can be
driven to **absolute zero**, forcing the Bayes Factor to crown the simpler model
regardless of how strong the quadratic effect is in the data.

#### 4. The Solution: Out-of-Sample PSIS-LOO
For predictive engineering decisions, we do not rely on Bayes Factors with
vague priors. We use **Pareto-Smoothed Importance Sampling Leave-One-Out
Cross-Validation (PSIS-LOO-CV)**. PSIS-LOO measures the model's ability to
predict holdout queries while auditing reliability via the shape parameter $k$.


## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Models: Compare polynomial regression models (Degrees 1, 2, 3, and 7) on
   noisy infrastructure telemetry.
2. Regularization: Place calibrated weakly informative priors beta_k ~ Normal(0, 1)
   to prevent Lindley's Paradox.
3. Metric: Evaluate out-of-sample expected log predictive density via PSIS-LOO with
   Pareto-k < 0.70 audit, rather than training R^2.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Evidence:      P(D | M) = integral P(D | theta, M) P(theta | M) dtheta
Out-of-sample: Compare Delta ELPD_LOO with standard error SE(Delta ELPD)
Audit:         Verify Pareto-k < 0.70 for all observations
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Bayesian Model Comparison Specialist. My team is arguing whether a
complex 7th-degree model is better than a quadratic model. Ask me 3 questions
about prior width, Lindley's Paradox, and out-of-sample LOO-CV before we decide."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
declarative_probabilistic_specification:
  module: "marginal_likelihood_and_model_comparison"
  domain: "database_latency_scaling"
  
  candidate_models:
    model_1_linear:
      formula: "y_i ~ Normal(alpha + beta_1 * x_i, sigma)"
      complexity: 1
    model_2_quadratic:
      formula: "y_i ~ Normal(alpha + beta_1 * x_i + beta_2 * x_i^2, sigma)"
      complexity: 2
    model_7_polynomial:
      formula: "y_i ~ Normal(alpha + sum_{k=1}^7 beta_k * x_i^k, sigma)"
      complexity: 7

  prior_regularization_contract:
    intercept: "Normal(mu=5.0, sigma=2.0)"
    linear_slope: "Normal(mu=0.0, sigma=2.0)"
    higher_order_terms: "Normal(mu=0.0, sigma=1.0)  # Regularizing prior!"
    noise_sigma: "Exponential(lam=1.0)"
    forbidden_priors:
      - "Uniform(-inf, inf)"
      - "Normal(0, 1000^2)  # Strictly banned to prevent Lindley's Paradox"

  evaluation_protocols:
    in_sample:
      compute_training_log_likelihood: true
      warning: "Never use for model selection"
    marginal_evidence:
      method: "Bridge Sampling / Laplace Approximation"
      prior_sensitivity_audit: "Evaluate BF_12 across sigma_0 in [0.5, 10.0]"
    out_of_sample_predictive:
      primary_metric: "PSIS-LOO ELPD (Expected Log Pointwise Predictive Density)"
      secondary_metric: "WAIC"
      diagnostics:
        pareto_k_threshold: 0.70
        assert_all_pareto_k_safe: true

  decision_contract:
    selection_rule: "Select Model M if ELPD_diff > 2 * SE_diff and max(k_i) < 0.7"
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

### Dataset Profile & Experimental Setup
* Ground Truth: $y = 5.0 + 2.0 x - 0.5 x^2 + \epsilon$,
  where $\epsilon \sim \mathcal{N}(0, 1.0)$.
* Observations: $N = 100$ points across $x \in [-3.0, 3.0]$.
* Candidate Models: Degree 1 (Linear), Degree 2 (Quadratic), Degree 3 (Cubic),
  Degree 7 (Polynomial).

```text
================================================================================
                    MODEL COMPARISON & PSIS-LOO TELEMETRY
================================================================================
Model         Train R^2   Train LogLik   Marginal LogLik    PSIS-LOO ELPD   SE(ELPD)   Bad k (>0.7)
------------------------------------------------------------------------------------------------
Degree 1         0.712        -168.4            -176.2           -174.1        8.4              0
Degree 2         0.941        -136.2            -143.5           -138.6        6.2              0  <-- WINNER!
Degree 3         0.943        -135.8            -147.1           -140.2        6.3              0
Degree 7         0.994        -112.4            -189.6           -184.2       14.8              8  <-- OVERFIT!
================================================================================
ELPD Difference (Degree 2 vs Degree 1): +35.5 +/- 5.8  (Decisive Predictive Win: > 6 SE)
ELPD Difference (Degree 2 vs Degree 3):  +1.6 +/- 1.1  (Parsimony Favors Degree 2)
ELPD Difference (Degree 2 vs Degree 7): +45.6 +/- 12.1 (Degree 7 Overfit Disaster)
================================================================================
```

---

### Visualization: Training Likelihood vs. True Predictive Generalization

```
       TRAINING LIKELIHOOD GREED VS. OUT-OF-SAMPLE REALITY
 Metric
   Value
    -100 |                                           * [Deg 7 Train LL: -112]
         |                          *       *
    -120 |                         [Deg 2] [Deg 3]
         |
    -140 |                         ●=======● [Deg 2 LOO: -138.6]
         |
    -160 |         * [Deg 1 Train: -168]
         |         ● [Deg 1 LOO: -174]
    -180 |-------------------------------------------● [Deg 7 LOO: -184.2]
         +---------------------------------------------------
                Deg 1           Deg 2           Deg 7
                Linear        Quadratic       Polynomial
         Legend: * In-Sample Training LL   ● Out-of-Sample LOO ELPD
```

### Telemetry Observations
1. **Degree 7 Collapse**: Despite achieving the highest in-sample training
   log-likelihood ($-112.4$), Degree 7 achieved the worst out-of-sample ELPD
   ($-184.2$) and triggered $8$ catastrophic Pareto-$k$ warnings ($k > 0.7$).
2. **Occam Penalty in Marginal Likelihood**: Marginal Log-Likelihood peaked
   sharply at Degree 2 ($-143.5$). Degree 7 was crushed ($-189.6$) by the Occam
   penalty over its 8-dimensional parameter volume.
3. **Decisive Calibration**: PSIS-LOO selects Degree 2 with an ELPD margin of
   $+35.5 \pm 5.8$ over Linear, and rejects Degree 3 via Occam parsimony.



## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned a clean, calibrated model comparison response: it
rejected training $R^2$ greed, audited out-of-sample expected log predictive
density (ELPD) via PSIS-LOO, identified high-leverage points with Pareto-$k$,
and applied Occam's parsimony to select the true quadratic generating model.

However, as an **Epistemic Architect**, you must never accept an AI agent's
model selection verdict at face value. How do you know the agent didn't fall
prey to Lindley's Paradox, hide infinite-variance importance weights, or select
an overparameterized cubic model when differences were mere sampling noise?

Before deploying model comparison results or automated model routing to
production, you audit the agent against **four mandatory verification gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Hierarchy shrink/pool invariants / PSIS Pareto-k < 0.7 / decay law │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Structural Invariants Verified]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Sensitivity & Lindley Paradox Audit (Domain Sanity Gate)

* **The Production Risk (Why This Gate Exists)**:
  In Bayesian model comparison using Bayes Factors or Marginal Likelihood
  $p(y \mid M)$, engineers often specify diffuse priors like
  $\beta \sim \mathcal{N}(0, 1000^2)$ under the mistaken assumption that they
  are "uninformative". Under marginal likelihood integration, diffuse priors
  trigger **Lindley's Paradox**: the prior volume penalty $\sigma_0^K$ scales
  with parameter dimension $K$, arbitrarily penalizing complex models. An
  automated AI agent relying on diffuse Bayes Factors will decisively choose a
  linear latency model over a true quadratic model ($BF_{01} = 2.4 \times 10^4$
  in favor of linear!). In production, deploying an underparameterized linear
  model causes server out-of-memory crashes because it fails to predict latency
  cliffs at high loads. Gate 1 guarantees calibrated, scale-aware priors.
* **First-Principles Mathematical Mechanism**:
  The marginal likelihood integrates likelihood over the prior parameter space:
  $$p(y \mid M) = \int_{\mathbb{R}^K} p(y \mid \theta, M) \,
  p(\theta \mid M) \, d\theta$$
  Under a Gaussian prior $\beta \sim \mathcal{N}(0, \sigma_0^2 I_K)$ in a
  linear regression model, as prior dispersion $\sigma_0 \to \infty$, the
  prior density behaves as $(2\pi \sigma_0^2)^{-K/2}$. The marginal
  likelihood scales asymptotically as:
  $$p(y \mid M_K) \propto \sigma_0^{-K}$$
  Comparing Model 1 ($K_1 = 2$) against Model 2 ($K_2 = 3$):
  $$BF_{12} = \frac{p(y \mid M_1)}{p(y \mid M_2)} \propto
  \frac{\sigma_0^{-2}}{\sigma_0^{-3}} = \sigma_0 \xrightarrow{\sigma_0
  \to \infty} \infty$$
  Arbitrarily large prior variance guarantees that the simpler model wins,
  regardless of empirical evidence!
  Gate 1 enforces standardized, weakly informative priors
  $\beta_k \sim \mathcal{N}(0, 1.0^2)$ for standardized predictors, bounding
  prior volume distortion.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Prior Scale Invariant: Prior standard deviation on standardized predictors
    must satisfy $\sigma_0 \le 2.50$ (`[PASS]`; diffuse priors
    $\sigma_0 \ge 10.0$ trigger `[FAIL: LINDLEY_PARADOX_VULNERABILITY]`).
  * Prior Predictive Variance: Prior predictive variance of predictions at
    domain bounds $x \in [-3, 3]$ satisfies $\text{Var}(y_{\text{sim}}) \le
    50.0$ (`[PASS]`).
  * Prior Sensitivity Derivative:
    $|\frac{\partial \log BF_{12}}{\partial \log \sigma_0}| \le 0.050$
    in the calibrated neighborhood (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Lindley Paradox & Prior Sensitivity Audit):
  Inspect priors across candidate models (Degrees 1, 2, 3, 7):
  1. Check prior standard deviation sigma_0 on regression coefficients.
  2. Verify sigma_0 <= 2.50 on standardized inputs.
  3. Flag [FAIL: LINDLEY_PARADOX_VULNERABILITY] if diffuse priors (sigma_0 >= 10) are used.
  4. Evaluate prior predictive variance of y_sim across x in [-3, 3].
  Assert priors are regularizing and scale-calibrated. Report [PASS] or [FAIL].
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) of Model Selection

* **The Production Risk (Why This Gate Exists)**:
  AI assistants frequently compute marginal likelihoods using pathological
  numerical routines, such as the harmonic mean estimator (notoriously known as
  "the worst method in statistics" due to infinite variance) or poorly tuned
  bridge sampling. If un-audited, the model comparison pipeline will select the
  wrong model architecture even on synthetic benchmarks where the true data
  generating process is known with mathematical certainty. Gate 2 evaluates
  model recovery across 5,000 synthetic simulations.
* **First-Principles Mathematical Mechanism**:
  SBC for model selection validates recovery across distinct generative
  realities:
  * Synthetic Universe A: $y \sim \text{Degree 1 (Linear)}$
  * Synthetic Universe B: $y \sim \text{Degree 2 (Quadratic)}$
  * Synthetic Universe C: $y \sim \text{Degree 3 (Cubic)}$
  Across $M = 5{,}000$ synthetic datasets generated from Universe B
  ($y = 5.0 + 2.0x - 0.5x^2 + \epsilon$, $\epsilon \sim \mathcal{N}(0, 1)$):
  1. Fit Degrees 1, 2, 3, and 7 using NUTS MCMC.
  2. Compute PSIS-LOO out-of-sample ELPD and bridge-sampled marginal likelihood.
  3. Verify that Degree 2 achieves higher ELPD than Degree 1 by at least
     $2 \times \text{SE}(\Delta \text{ELPD})$.
  4. Verify that Degree 2 is chosen over Degree 3 via parsimony.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Synthetic Architecture Recovery: Degree 2 model is correctly selected in
    $\ge 95.0\%$ of Universe B synthetic trials (`[PASS]`).
  * Banned Estimator Check: Harmonic mean estimator is strictly absent from
    the codebase (`[PASS]`; presence triggers
    `[FAIL: INFINITE_VARIANCE_ESTIMATOR]`).
  * Bridge Sampling Stability: Relative standard error of bridge sampling
    marginal log-likelihood $\le 0.010$ across independent runs (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration of Model Comparison):
  Generate 5,000 synthetic datasets from known quadratic truth (N=100, sigma=1.0):
  For each dataset:
    1. Fit Degrees 1, 2, 3, and 7 models.
    2. Compute PSIS-LOO ELPD and standard errors.
    3. Evaluate model selection rule (ELPD difference > 2*SE, parsimony tie-breaker).
  Assert that Degree 2 is selected in >= 95.0% of trials.
  Assert harmonic mean estimator is not present.
  Report [PASS] or [FAIL] with synthetic recovery rate.
  ```

---

### Gate 3: Structural Invariant: PSIS Pareto-k Reliability Diagnostic

* **The Production Risk (Why This Gate Exists)**:
  Leave-one-out cross-validation without costly refitting relies on importance
  sampling weights $w_i^{(s)} = \frac{1}{p(y_i \mid \theta^{(s)})}$. When a
  model is overparameterized (e.g. Degree 7 polynomial), high-leverage points
  cause the importance ratio distribution to develop an extremely heavy right
  tail. The variance of the importance weights becomes infinite, causing
  PSIS-LOO ELPD estimates to fluctuate wildly and rendering cross-validation
  metrics entirely untrustworthy. An uncalibrated AI agent will quote a glowing
  ELPD score without checking Pareto-$k$, deploying an unstable model that
  collapses on production anomalies.
* **First-Principles Mathematical Mechanism**:
  Vehtari, Gelman, and Gabry (2017) Pareto-Smoothed Importance Sampling (PSIS):
  Fit a Generalized Pareto Distribution to the largest
  $M = \min(S/5, 3\sqrt{S})$ importance ratios:
  $$p(w \mid k, \sigma) = \frac{1}{\sigma}\left(1 + k \frac{w -
  u}{\sigma}\right)^{-\frac{1}{k} - 1}$$
  The shape parameter $k$ governs the existence of moments:
  * $k < 0.50$: All moments exist; Central Limit Theorem holds rapidly;
    importance weights are highly reliable.
  * $0.50 \le k < 0.70$: Variance is finite; convergence is slower but
    acceptable for decision-making.
  * $k \ge 0.70$: Variance of importance weights is infinite; CLT breaks down;
    PSIS-LOO estimates are untrustworthy and must trigger exact refitting.
  Telemetry Audit:
  * Degree 2 model: $100\%$ of observations have $k_i < 0.52$ (zero bad points).
  * Degree 7 model: $8$ observations have $k_i > 0.70$ (max $k = 1.14$),
    disqualifying it from unadjusted comparison.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Winning Model Pareto-$k$ Invariant: For the selected model,
    $\max_{i=1}^N k_i < 0.70$ strictly (`[PASS]`; any point $k_i \ge 0.70$
    triggers `[FAIL: UNRELIABLE_PSIS_LOO]`).
  * Overfit Detection: Degree 7 model flags $\ge 5$ observations with
    $k_i > 0.70$ (`[PASS]`).
  * Effective Sample Size of Weights: $\text{ESS}_{\text{PSIS}} \ge 100$ for
    all $N = 100$ observations (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (PSIS Pareto-k Diagnostic & Structural Reliability):
  Extract pointwise Pareto shape parameters k_i for all candidate models:
  1. Inspect distribution of k_i for winning model.
  2. Assert 100% of observations satisfy k_i < 0.70.
  3. Flag [FAIL: UNRELIABLE_PSIS_LOO] if max(k_i) >= 0.70 without refitting.
  4. Count high-influence outliers in Degree 7 polynomial.
  Report [PASS] with max k_i and outlier count or [FAIL].
  ```

---

### Gate 4: Out-of-Sample Predictive Difference & Stacking Loss Gate

* **The Production Risk (Why This Gate Exists)**:
  A naive AI assistant relies on in-sample training metrics, recommending the
  Degree 7 polynomial because $R^2 = 0.994$ and training log-likelihood is
  $-112.4$ (vastly higher than Quadratic $R^2 = 0.941$, LL = $-136.2$). In
  production, Degree 7 exhibits wild Runge-phenomenon oscillations between data
  points, making disastrous predictions. Alternatively, the AI declares Degree 3
  the winner over Degree 2 based on a tiny raw ELPD advantage
  ($\Delta \text{ELPD} = +1.6 \pm 1.1$), adding unnecessary complexity that
  slows inference latency. Gate 4 guarantees calibrated model selection via
  generalization bounds and stacking loss.
* **First-Principles Mathematical Mechanism**:
  We evaluate expected log predictive density difference and its standard error:
  $$\Delta \text{elpd}_{A - B} = \text{elpd}_{\text{loo}}^{(A)} -
  \text{elpd}_{\text{loo}}^{(B)}$$
  $$\text{SE}(\Delta \text{elpd}_{A - B}) = \sqrt{N \cdot
  \widehat{\text{Var}}_i\left(\text{elpd}_{\text{loo}, i}^{(A)} -
  \text{elpd}_{\text{loo}, i}^{(B)}\right)}$$
  Decision Protocol:
  1. **Decisive Superiority**: Model A beats Model B if:
     $$\Delta \text{elpd}_{A - B} > 2 \cdot \text{SE}(\Delta \text{elpd})$$
     For Degree 2 vs Degree 1: $\Delta \text{ELPD} = +35.5 \pm 5.8$. Since
     $35.5 > 2(5.8) = 11.6$, Degree 2 is decisively superior ($> 6\text{ SE}$).
  2. **Parsimony Tie-Breaker**: If $|\Delta \text{elpd}| \le 2 \cdot \text{SE}$,
     Occam's razor selects the model with fewer effective parameters
     $p_{\text{loo}}$.
     For Degree 2 vs Degree 3: $\Delta \text{ELPD} = +1.6 \pm 1.1$. Since
     $1.6 < 2(1.1) = 2.2$, the difference is indistinguishable from noise.
     Parsimony selects Degree 2 ($p_{\text{loo}} = 3.2$ vs $4.8$).
  3. **Overfit Rejection**: Degree 7 yields $\Delta \text{ELPD} = -45.6 \pm
     12.1$ relative to Degree 2, confirming catastrophic overfitting.
  4. **Bayesian Model Stacking**: Optimal stacking weights
     $w^* = \arg\max_w \sum_{i=1}^N \log \sum_{k=1}^K w_k p(y_i \mid y_{-i},
     M_k)$ assign $w_{\text{Deg 2}} = 0.88$ and $w_{\text{Deg 3}} = 0.12$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Decisive Win Assertion: $\Delta \text{ELPD}(\text{Deg 2} - \text{Deg 1})
    \ge 2 \cdot \text{SE}$ (`[PASS]`).
  * Parsimony Selection Invariant: When $|\Delta \text{ELPD}| < 2 \cdot
    \text{SE}$, the simpler model is selected (`[PASS]`; picking Degree 3
    triggers `[FAIL: PARSIMONY_VIOLATION]`).
  * Overfit Rejection: Degree 7 rejected with $\Delta \text{ELPD} \le -20.0$
    (`[PASS]`).
  * Stacking Weight Invariant: Winning model stacking weight $w^* \ge 0.70$
    (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Out-of-Sample ELPD Difference & Stacking Decision):
  Extract PSIS-LOO table for candidate models:
  1. Compute Delta ELPD and SE(Delta ELPD) between Degree 2 and Degree 1.
  2. Assert Degree 2 outperforms Degree 1 by > 2 * SE.
  3. Compare Degree 2 and Degree 3: if |Delta ELPD| <= 2 * SE, enforce parsimony (select Degree 2).
  4. Flag [FAIL: TRAINING_GREED] if selection is based on training R^2 or training log-likelihood.
  5. Compute Bayesian stacking weights and assert Degree 2 weight >= 0.70.
  Report [PASS] or [FAIL] with model comparison decision log.
  ```



## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Comparison Dimension | Naive AI Assistant Output | Calibrated Probabilistic Specification |
| :--- | :--- | :--- |
| **Primary Metric** | In-sample $R^2$ or training log-likelihood | Out-of-sample PSIS-LOO ELPD and WAIC |
| **Model Recommendation**| Recommends Degree 7 (overfits training noise) | Selects Degree 2; penalizes unneeded complexity |
| **Prior Specification** | Inserts diffuse priors $\mathcal{N}(0, 1000^2)$ | Enforces regularizing priors $\mathcal{N}(0, 1.0)$ |
| **Hypothesis Testing** | Triggers Lindley's Paradox; spurious null result | Calibrated Bayes Factors with sensitivity audits |
| **Diagnostic Auditing** | No checks on importance sampling stability | Enforces Pareto-$k < 0.70$ on all observations |
| **Decision Standard** | Picks highest point metric | Requires $\Delta\text{ELPD} > 2 \times \text{SE}$ |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

Test your skills in auditing AI-generated model comparison workflows.

---

### 🟢 Tier 1: Spot the AI Hallucination
An AI agent evaluates two competing fraud detection models:
* Model A: Simple Logistic Regression ($R^2 = 0.81$, LOO ELPD = $-410$).
* Model B: 15-layer Deep Neural Net ($R^2 = 0.99$, LOO ELPD = $-620$,
  45 points with $k > 0.7$).

The AI concludes:
> *"Model B should be deployed immediately because its training accuracy is
> 99%, compared to only 81% for Model A."*

**Task**: Identify the fatal error in the AI's deployment recommendation.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
The agent committed the cardinal sin of model comparison: deploying based on
in-sample training fit ($R^2$) rather than out-of-sample predictive performance.

Model B's out-of-sample LOO ELPD is $-620$, drastically worse than Model A's
$-410$ ($\Delta\text{ELPD} = -210$). Furthermore, 45 observations have Pareto
$k > 0.7$, proving that Model B is brittle and hypersensitive to training
outliers. Model B has memorized noise and will suffer catastrophic failure in
production. Model A is the objectively superior deployment choice.
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An engineer prompted an agent:
> *"Compute the Bayes Factor between Model 1 (null hypothesis $\beta = 0$) and
> Model 2 (free slope $\beta$), using an uninformative prior $\beta \sim
> \text{Uniform}(-10^6, 10^6)$."*

**Task**: Explain why this triggers Lindley's Paradox, and write a calibrated
specification using both regularized Bayes Factors and PSIS-LOO.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
Setting a uniform prior over $[-10^6, 10^6]$ spreads the prior probability
density to $1 / (2 \times 10^6) = 5 \times 10^{-7}$. The Occam factor scales
linearly with prior density, driving the marginal likelihood $P(\mathcal{D} \mid
\mathcal{M}_2)$ to near zero. Model 1 will win the Bayes Factor test by a factor
of thousands even if the true $\beta = 4.2$ with overwhelming statistical
evidence (Lindley's Paradox).

**Calibrated Specification**:

```yaml
calibrated_model_comparison:
  model_1_null: "y ~ Normal(alpha, sigma)"
  model_2_slope: "y ~ Normal(alpha + beta * x, sigma)"
  calibrated_priors:
    alpha: "Normal(mu=0, sigma=5)"
    beta: "Normal(mu=0, sigma=2.0)  # Calibrated physical scale"
    sigma: "Exponential(lam=1.0)"
  evaluation:
    bayes_factor:
      method: "Savage-Dickey Density Ratio or Bridge Sampling"
      audit_scales: [0.5, 1.0, 2.0, 5.0]
    cross_validation:
      method: "PSIS-LOO"
      assert_max_pareto_k: 0.70
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
You are designing an autonomous ML selection agent that evaluates five candidate
regression architectures for dynamic pricing. The agent must automatically rule
out overfit models, check Pareto-$k$ diagnostics, and select the simplest model
within 1 standard error of the best predictor.

**Task**: Write the complete declarative agent contract.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:

```yaml
autonomous_agent_contract:
  role: "Predictive Model Selection & Parsimony Agent"
  objective: "Select optimal pricing model balancing predictive accuracy and simplicity"
  
  candidate_set:
    models: ["linear", "interaction", "spline_df4", "spline_df10", "ensemble"]
    priors: "Weakly informative regularizing priors on all coefficients"
    
  diagnostic_gates:
    gate_1_pareto_k:
      condition: "All observations must have Pareto k_i < 0.70"
      on_violation: "Flag model as unreliable; do not select"
    gate_2_generalization:
      metric: "PSIS-LOO ELPD"
      ranking: "Sort candidate models by ELPD descending"
      
  parsimony_decision_rule:
    algorithm:
      step_1: "Identify model M_best with highest ELPD"
      step_2: "Compute threshold: T_elpd = ELPD(M_best) - SE_diff(M_best, M_candidate)"
      step_3: "Filter all models with ELPD >= T_elpd and valid Pareto k"
      step_4: "Among filtered set, select model with minimum parameter count"
      step_5: "Return selected model and publish ELPD telemetry report"
```
</details>


## Key Takeaways & Course Navigation

### Core Architectural Rules for AI Model Selection
1. **Never Select Models by In-Sample Fit**: High $R^2$ and high training
   likelihood are symptoms of parameter greed. Unconstrained models memorize
   noise and degrade in production.
2. **Beware Lindley's Paradox in Bayes Factors**: Diffuse "uninformative"
   priors dilute the probability budget across infinite space, spuriously
   penalizing real effects. Always audit prior sensitivity or rely on PSIS-LOO.
3. **Use PSIS-LOO as the Gold Standard**: Leave-one-out predictive density
   measures real generalization without sensitivity to arbitrary prior bounds.
4. **Always Audit Pareto-$k$ Diagnostics**: If $\max(k_i) > 0.70$, the
   importance weights are unstable, indicating influential outliers or model
   misspecification.

---

### 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](../README.md)**
* ⬅️ **Previous**: **[Ch 13: Hierarchical Models & Shrinkage](13_hierarchical_models_and_partial_pooling.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* ➡️ **Next**: **[Ch 15: Dynamic Memory & Decay (γ)](15_calibrating_bayesian_decay_and_memory.ipynb)** *(Part IV: Dynamic Systems, Decision Theory & Predictive Guarantees)*
* 🔄 **Companion Tracks for Chapter 14**: [📖 Conceptual](../conceptual/14_marginal_likelihood_and_model_comparison.ipynb) | [🤖 AI Track](14_marginal_likelihood_and_model_comparison.ipynb) | [🐍 Python](../python/14_marginal_likelihood_and_model_comparison.ipynb) | [🔵 R](../r/14_marginal_likelihood_and_model_comparison.ipynb)

